In [2]:
from langgraph.graph import StateGraph, START, END
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_mistralai import ChatMistralAI, MistralAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langgraph.graph import StateGraph, START, END
from typing import List, TypedDict, Literal
from langchain_core.documents import Document
from pydantic import BaseModel, Field
import time


from dotenv import load_dotenv
load_dotenv()

True

In [3]:
docs = (
    PyPDFLoader("./documents/Company_Policies.pdf").load()
    + PyPDFLoader("./documents/Company_Profile.pdf").load()
    + PyPDFLoader("./documents/Product_and_Pricing.pdf").load()
)

In [4]:
chunks = RecursiveCharacterTextSplitter(
    chunk_size=600, chunk_overlap=150
).split_documents(docs)

In [5]:
embeddings = MistralAIEmbeddings(model="text-embedding-3-large")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 4})

/Users/mayankmokta/Documents/PYTHON/Agentic_AI/Self-RAG/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
An error occurred with MistralAI
Traceback (most recent call last):
  File "/Users/mayankmokta/Documents/PYTHON/Agentic_AI/Self-RAG/.venv/lib/python3.11/site-packages/langchain_mistralai/embeddings.py", line 277, in embed_documents
    batch_responses = [
                      ^
  File "/Users/mayankmokta/Documents/PYTHON/Agentic_AI/Self-RAG/.venv/lib/python3.11/site-packages/langchain_mistralai/embeddings.py", line 278, in <listcomp>
    _embed_batch(batch) for batch in self._get_batches(texts)
    ^^^^^^^^^^^^^^^^^^^
  File "/Users/mayankmokta/Documents/PYTHON/Agentic_AI/Self-RAG/.venv/lib/python3.11/site-packages/tenacity/__init__.py", line 331, in wrapped_f
    return copy(f, *args, **kw)
  

HTTPStatusError: Client error '400 Bad Request' for url 'https://api.mistral.ai/v1/embeddings'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/400

In [ ]:
class State(TypedDict):
    question: str
    need_retrieval: bool
    docs: List[Document]
    answer: str

In [ ]:
llm = ChatMistralAI()

In [ ]:
class RetrieveDecision(BaseModel):
    should_retrieve: bool = Field(
        ...,
        description="True if external documents are needed to answer reliably, else False."
    )

decide_retrieval_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You decide whether retrieval is needed.\n"
            "Return JSON that matches this schema:\n"
            "{{'should_retrieve': boolean}}\n\n"
            "Guidelines:\n"
            "- should_retrieve=True if answering requires specific facts, citations, or info likely not in the model.\n"
            "- should_retrieve=False for general explanations, definitions, or reasoning that doesn't need sources.\n"
            "- If unsure, choose True."
        ),
        ("human", "Question: {question}"),
    ]
)

should_retrieve_llm = decide_retrieval_prompt | llm.with_structured_output(RetrieveDecision)

def decide_retrieval(state: "State"):
    q = state["question"]
    answer = should_retrieve_llm.invoke({
        "question": 1
    })
    return {"need_retrieval" : answer.should_retrieve}


In [ ]:
direct_generation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using only your general knowledge.\n"
            "Do NOT assume access to external documents.\n"
            "If you are unsure or the answer requires specific sources, say:\n"
            "'I don't know based on my general knowledge.'"
        ),
        ("human", "{question}"),
    ]
)

def generate_direct(state: State):
    q = state["question"]
    answer = (direct_generation_prompt | llm).invoke({"question": q})
    return {"answer": answer.content}

In [ ]:
def retrieve(state: State):
    q = state['question']
    docs = retriever.invoke({"query": q})
    return {"docs": docs}


In [ ]:
def route_after_decide(state: State) -> Literal["generate_direct", "retrieve"]:
    if state["need_retrieval"] == True:
        return "retrieve"
    return "generate_direct"

In [ ]:
g = StateGraph(State)

g.add_node("decide_retrieval", decide_retrieval)
g.add_node("generate_direct", generate_direct)
g.add_node("retrieve", retrieve)

g.add_edge(START, "decide_retrieval")
g.add_conditional_edges("decide_retrieval", route_after_decide, {
    "retrieve": "retrieve", "generate_direct": "generate_direct"
})
g.add_edge("retrieve", END)
g.add_edge("generate_direct", END)

app = g.compile()
app